# Sanity checks: metrics 1 and 2

Hand checks for `src/metrics.py`, plus both versions of each open decision so they can be compared. Definitions are in `METHOD.md`.

In [1]:
import sys; sys.path.insert(0, '..')
import pandas as pd
from src.metrics import load, retention_rows, retention, corner_deficit_rows, corner_deficit
pd.set_option('display.precision', 2)
results = load('results', {'R', 'S'})
corners = load('corners', {'Q', 'SQ'})
print(results.groupby(['SessionType'])['Round'].nunique().to_dict(), 'rounds by session type')

{'GP': 16, 'Sprint': 5} rounds by session type


## Metric 1: position retention

### Hand check: Azerbaijan GP (Round 15)
Expected from the official result: Antonelli 16th → 5th (+11), Verstappen 8th → 2nd (+6), Piastri 3rd → 13th (−10). Retired cars (e.g. Norris, both Aston Martins) should be missing; Bottas was "Retired" but classified 16th, so he should be included.

In [2]:
rows = retention_rows(results)
rows[(rows['Round'] == 15) & (rows['Session'] == 'R')].sort_values('Finish')

,Round,EventName,Session,SessionType,TeamName,Abbreviation,GridPosition,Finish,Gained,Lost,Expected,VsExpected
418,15,Azerbaijan Grand Prix,R,GP,Mercedes,RUS,1.0,1.0,0.0,0.0,-1.15,1.15
419,15,Azerbaijan Grand Prix,R,GP,Red Bull Racing,VER,8.0,2.0,6.0,0.0,-0.45,6.45
420,15,Azerbaijan Grand Prix,R,GP,Red Bull Racing,HAD,4.0,3.0,1.0,0.0,-0.69,1.69
421,15,Azerbaijan Grand Prix,R,GP,Ferrari,LEC,2.0,4.0,-2.0,-2.0,-1.15,-0.85
422,15,Azerbaijan Grand Prix,R,GP,Mercedes,ANT,16.0,5.0,11.0,0.0,2.35,8.65
423,15,Azerbaijan Grand Prix,R,GP,Ferrari,HAM,6.0,6.0,0.0,0.0,0.61,-0.61
424,15,Azerbaijan Grand Prix,R,GP,Racing Bulls,LIN,15.0,7.0,8.0,0.0,2.35,5.65
425,15,Azerbaijan Grand Prix,R,GP,Haas F1 Team,OCO,13.0,8.0,5.0,0.0,1.83,3.17
426,15,Azerbaijan Grand Prix,R,GP,Haas F1 Team,BEA,10.0,9.0,1.0,0.0,-0.24,1.24
427,15,Azerbaijan Grand Prix,R,GP,Williams,SAI,14.0,10.0,4.0,0.0,1.83,2.17


### Hand check: one team computed by hand
Mercedes' net score should equal the plain average of its rows.

In [3]:
merc = rows[rows['TeamName'] == 'Mercedes']
print('by hand:', round(merc['Gained'].mean(), 3), '| function:', round(retention(results).loc['Mercedes', 'score'], 3), '| n =', len(merc))

by hand: -0.3 | function: -0.3 | n = 40


### Decision: how to count places
- **net**: average places gained (gains and losses both count).
- **lost**: average places lost (gains count as 0).
- **adjusted**: places gained compared with the typical car starting from the same grid slot (slots pooled in pairs: P1–2, P3–4, ...).

Higher is better in all three. Compare each with the team's average starting position: net and lost mostly track the grid (a car starting 20th can only gain, and gets promoted when cars ahead retire). Adjusted removes that built-in effect.

In [4]:
comp = pd.DataFrame({m: retention(results, m)['score'] for m in ['net', 'lost', 'adjusted']})
comp['avg_grid'] = rows.groupby('TeamName')['GridPosition'].mean()
comp = comp.sort_values('adjusted', ascending=False)
print('Spearman correlation with average grid position:', comp.corr(method='spearman')['avg_grid'].round(2).drop('avg_grid').to_dict())
comp

Spearman correlation with average grid position: {'net': 0.87, 'lost': 0.86, 'adjusted': -0.79}


,net,lost,adjusted,avg_grid
TeamName,,,,
Racing Bulls,2.05,-0.50,1.28,11.63
Red Bull Racing,0.81,-1.19,0.57,7.59
Ferrari,0.03,-0.72,0.36,4.41
McLaren,0.06,-0.97,0.31,4.94
Mercedes,-0.30,-1.48,0.24,3.50
Alpine,1.18,-0.69,0.07,11.95
Haas F1 Team,1.63,-0.39,-0.21,14.53
Audi,0.64,-0.82,-0.44,12.36
Aston Martin,3.21,0.00,-0.51,19.08


### Expected places gained by starting slot
The baseline for **adjusted**: the average places gained by all drivers starting in each pair of grid slots, across every race and Sprint.

In [5]:
rows.groupby((rows['GridPosition'] - 1) // 2 * 2 + 1)['Expected'].first().rename('expected gain').rename_axis('grid slots from').round(1)

grid slots from
1.0    -1.2
3.0    -0.7
5.0     0.6
7.0    -0.4
9.0    -0.2
11.0    0.7
13.0    1.8
15.0    2.4
17.0    2.1
19.0    3.7
21.0    5.5
Name: expected gain, dtype: float64

### Sprints vs Grand Prix (net)
Do teams behave differently in Sprints? (Sprints have about 22 rows per round vs about 22 per GP, but only 5 rounds.)

In [6]:
rows.pivot_table(index='TeamName', columns='SessionType', values='Gained', aggfunc='mean').sort_values('GP', ascending=False)

SessionType,GP,Sprint
TeamName,,
Aston Martin,4.00,1.89
Cadillac,2.80,0.44
Racing Bulls,2.10,1.88
Red Bull Racing,1.96,-2.30
Williams,1.67,1.80
Haas F1 Team,1.61,1.70
Alpine,1.41,0.50
Audi,1.00,-1.00
McLaren,0.04,0.10


## Metric 2: slow-corner speed

### Data quality: frozen speed readings
The 2026 speed channel sometimes stalls for up to ~1 s, holding one value (e.g. 305 km/h straight through a braking zone). Corner readings with 3+ identical consecutive speed samples are dropped, as are readings more than 30 km/h from the other teams at that corner. Corners are located by track-map position (X/Y), because FastF1's distance-along-lap drifts by up to ~3%.

If the dropped share is similar across teams, the cleaning doesn't favour anyone.

In [7]:
from src.metrics import clean_corner_speeds
cl = clean_corner_speeds(corners, 'Q1')
print(f"dropped overall: {(~cl['Valid']).mean():.1%}")
(~cl['Valid']).groupby(cl['Team']).mean().mul(100).round(1).sort_values().rename('dropped %')

dropped overall: 9.8%

Team
Cadillac            6.9
Haas F1 Team        7.6
Ferrari             7.9
Racing Bulls        8.3
Alpine              8.6
Williams            8.9
McLaren             9.6
Audi               11.2
Aston Martin       12.2
Mercedes           13.2
Red Bull Racing    13.2
Name: dropped %, dtype: float64

### Hand check: Azerbaijan qualifying, whole-session fastest lap
The Oct 3 test (before the cleaning) gave Mercedes 2.4 km/h behind the best team in each slow corner → Cadillac 8.5. The exact values change with the cleaning, but the order should be broadly similar: Mercedes near the front, Cadillac, Williams and Aston Martin near the back.

In [8]:
dr_all = corner_deficit_rows(corners, 'all')
dr_all[(dr_all['Round'] == 15)].sort_values('Deficit')[['Team', 'Deficit', 'SlowCorners', 'ValidCorners']]

,Team,Deficit,SlowCorners,ValidCorners
172,Mercedes,2.25,13,13
169,Ferrari,2.65,13,13
171,McLaren,3.69,13,13
170,Haas F1 Team,3.77,13,13
165,Alpine,4.00,13,13
167,Audi,4.31,13,13
174,Red Bull Racing,5.92,13,12
173,Racing Bulls,6.08,13,13
175,Williams,6.23,13,13
166,Aston Martin,6.69,13,13


### Slow corners found per session
The data picks the slow corners (median minimum speed below 120 km/h). Monza (Round 13) is a high-speed track, so few slow corners are expected there.

In [9]:
dr_q1 = corner_deficit_rows(corners, 'Q1')
dr_q1.drop_duplicates(['Round', 'Session'])[['Round', 'EventName', 'Session', 'SlowCorners']]

,Round,EventName,Session,SlowCorners
0,1,Australian Grand Prix,Q,3
11,2,Chinese Grand Prix,Q,7
22,2,Chinese Grand Prix,SQ,4
33,3,Japanese Grand Prix,Q,3
44,4,Miami Grand Prix,Q,9
55,4,Miami Grand Prix,SQ,9
66,5,Canadian Grand Prix,Q,6
77,5,Canadian Grand Prix,SQ,6
88,6,Monaco Grand Prix,Q,12
99,7,Barcelona Grand Prix,Q,3


### Decision: fastest lap from Q1 only vs the whole session
- **Q1**: every car runs it, all on the same tyre (Softs in Qualifying, Mediums in Sprint Qualifying), with similar track conditions.
- **all**: each team's best lap overall. Top teams set it later in the session on a faster, rubbered-in track, and in Sprint Qualifying the final round switches to Softs, so top teams get a tyre advantage.

Lower deficit is better.

In [10]:
cd = corner_deficit(corners, 'Q1').rename(columns={'score': 'Q1'}).join(corner_deficit(corners, 'all')['score'].rename('all'))
cd['Q1_rank'] = cd['Q1'].rank().astype(int)
cd['all_rank'] = cd['all'].rank().astype(int)
cd

,Q1,n,all,Q1_rank,all_rank
Team,,,,,
Ferrari,2.88,18,2.99,1,1
Mercedes,4.01,18,3.85,2,3
Audi,4.07,18,4.91,3,7
McLaren,4.18,18,3.70,4,2
Alpine,4.30,18,4.51,5,5
Haas F1 Team,4.82,18,5.68,6,8
Red Bull Racing,4.95,18,4.27,7,4
Racing Bulls,5.23,18,4.91,8,6
Aston Martin,5.76,18,7.00,9,9


### Consistency across sessions (Q1)
Is a team's deficit stable from track to track, or driven by one or two sessions?

In [11]:
dr_q1.pivot_table(index='Team', columns='Round', values='Deficit').round(1)

Round,1,2,3,4,5,6,7,9,10,11,12,13,15
Team,,,,,,,,,,,,,
Alpine,3.7,6.3,0.0,4.5,3.5,3.6,3.0,5.4,4.0,6.8,5.2,3.0,3.7
Aston Martin,6.7,4.0,8.0,6.0,6.4,5.4,5.3,5.3,7.2,7.7,5.6,2.5,6.2
Audi,4.3,3.8,6.0,3.8,9.7,3.4,3.3,1.4,1.8,4.5,3.2,2.0,3.8
Cadillac,8.0,6.2,6.0,7.0,7.6,3.6,5.0,6.9,3.8,5.8,5.0,4.0,6.6
Ferrari,0.5,1.8,1.0,3.3,6.3,4.1,2.7,1.8,3.8,1.5,2.3,2.0,5.3
Haas F1 Team,4.0,3.0,4.3,3.2,7.9,5.3,1.0,4.3,5.1,6.3,7.2,4.5,4.9
McLaren,2.3,4.0,NaN,4.5,6.5,2.6,3.7,5.6,4.3,2.0,3.9,3.0,4.1
Mercedes,2.5,6.1,2.3,4.3,7.5,4.9,0.7,4.4,5.2,2.7,3.5,2.0,2.4
Racing Bulls,4.7,5.4,4.4,4.8,8.8,3.8,4.3,4.0,6.0,2.7,5.2,4.0,7.8


## Metric 3: tyre degradation

One row per stint: the slope of lap time against tyre age, and that slope relative to the field average for the same session and compound (`RelSlope`, s/lap; lower is better). The GP races at Canada (Round 5) and Kuala Lumpur (Round 16) are excluded by default because both were mixed-weather, and a drying track gets faster lap by lap, which disguises wear.

In [12]:
from src.metrics import stint_rows, tyre_deg, clean_stint_laps
import numpy as np
laps = load('laps', {'R', 'S'})
st = stint_rows(laps)
print(len(st), 'stints;', st['SessionType'].value_counts().to_dict())
st['Slope'].describe().round(3)

718 stints; {'GP': 619, 'Sprint': 99}


count    7.18e+02
mean     7.00e-03
std      9.20e-02
min     -3.75e-01
25%     -3.90e-02
50%      6.00e-03
75%      5.20e-02
max      6.64e-01
Name: Slope, dtype: float64

### Why most raw slopes are near zero or negative
The car burns roughly 1–2 kg of fuel per lap and gets lighter, so it speeds up through a stint; tyre wear slows it down. The raw slope is the net of the two. Comparing each stint with the field in the same session and compound cancels the fuel effect, which is roughly the same for every car.

### Hand check: one stint
The slope from the function should equal the textbook least-squares formula computed by hand.

In [13]:
row = st.sort_values('Laps', ascending=False).iloc[0]
cl = clean_stint_laps(laps)
g = cl[(cl.Round == row.Round) & (cl.Session == row.Session) & (cl.Driver == row.Driver) & (cl.Stint == row.Stint)]
x, y = g['TyreLife'].to_numpy(float), g['LapS'].to_numpy()
manual = ((x - x.mean()) * (y - y.mean())).sum() / ((x - x.mean()) ** 2).sum()
print(f"R{row.Round} {row.EventName} {row.Driver} stint {int(row.Stint)} ({row.Compound}, {len(g)} laps): function {row.Slope:.4f}, by hand {manual:.4f}")

R6 Monaco Grand Prix LIN stint 1 (MEDIUM, 56 laps): function 0.0259, by hand 0.0259


### Team scores, with uncertainty
`se` is the standard error of each team's mean. Two teams whose scores differ by less than about 2 × se can't really be told apart.

In [14]:
td = st.groupby('Team')['RelSlope'].agg(score='mean', se=lambda s: s.std() / np.sqrt(len(s)), n='size').sort_values('score')
td

,score,se,n
Team,,,
Mercedes,-1.51e-02,7.20e-03,73
Racing Bulls,-1.21e-02,6.33e-03,65
Audi,-7.31e-03,1.15e-02,64
Red Bull Racing,-1.97e-03,7.12e-03,63
Williams,-1.87e-03,1.16e-02,67
McLaren,-1.52e-03,5.00e-03,66
Alpine,1.54e-03,6.26e-03,69
Ferrari,3.14e-03,5.73e-03,74
Haas F1 Team,5.49e-03,8.98e-03,66


### Sensitivity: mixed-weather races and Sprints
Ranks under three versions. If a team's rank barely moves, its score is robust.

In [15]:
sens = pd.DataFrame({
    'default': tyre_deg(laps)['score'],
    'incl_mixed_weather': tyre_deg(laps, exclude_gp_rounds=[])['score'],
    'gp_only': tyre_deg(laps[laps['Session'] == 'R'])['score'],
}).rank().astype(int).sort_values('default')
sens

,default,incl_mixed_weather,gp_only
Team,,,
Mercedes,1,3,1
Racing Bulls,2,1,3
Audi,3,5,2
Red Bull Racing,4,7,4
Williams,5,4,8
McLaren,6,2,5
Alpine,7,6,7
Ferrari,8,10,6
Haas F1 Team,9,9,9


### Sensitivity: Sprints in the other two metrics
Ranks with and without Sprint sessions.

In [16]:
sens2 = pd.DataFrame({
    'retention_all': retention(results, 'adjusted')['score'].rank(ascending=False),
    'retention_gp_only': retention(results[results['Session'] == 'R'], 'adjusted')['score'].rank(ascending=False),
    'corner_all': corner_deficit(corners, 'Q1')['score'].rank(),
    'corner_gp_only': corner_deficit(corners[corners['Session'] == 'Q'], 'Q1')['score'].rank(),
}).astype(int)
sens2

,retention_all,retention_gp_only,corner_all,corner_gp_only
Alpine,6,6,5,5
Aston Martin,9,9,9,11
Audi,8,7,3,3
Cadillac,11,11,10,9
Ferrari,3,3,1,1
Haas F1 Team,7,8,6,6
McLaren,4,5,4,4
Mercedes,5,4,2,2
Racing Bulls,1,2,8,8
Red Bull Racing,2,1,7,7
